# Motorsports Data Analysis Starter

This notebook demonstrates the core data manipulation and visualization tools available in this environment.

## Libraries Included

- **pandas** - Tabular data manipulation
- **numpy** - Numeric computing
- **pyarrow** - Efficient columnar data format (Parquet)
- **matplotlib** - Static plotting
- **seaborn** - Statistical visualization
- **plotly** - Interactive plots

In [1]:
# Import core libraries
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Import libxrk:
from libxrk import aim_xrk


# Configure matplotlib for better inline display
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ All libraries imported successfully")
print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")
print(f"pyarrow version: {pa.__version__}")

✓ All libraries imported successfully
pandas version: 2.3.3
numpy version: 2.3.4
pyarrow version: 22.0.0


In [2]:
log = aim_xrk("CMD_Inferno 86_Fuji GP Sh_Generic testing_a_2248.xrk")
channels = log.get_channels_as_table().to_pandas()
laps = log.laps.to_pandas()

/home/m3rlin45/.cache/pypoetry/virtualenvs/motorsports-data-notebook-qUYIIvW5-py3.12/lib/python3.12/site-packages/libxrk/gps.py:334: RuntimeWarning: invalid value encountered in divide
  t = np.maximum(-np.sum(SN * O, axis=1) / np.sum(SN * D, axis=1), 0)
/home/m3rlin45/.cache/pypoetry/virtualenvs/motorsports-data-notebook-qUYIIvW5-py3.12/lib/python3.12/site-packages/libxrk/gps.py:347: RuntimeWarning: divide by zero encountered in divide
  t = np.maximum(-np.sum(SN * O, axis=1) / np.sum(SN * D, axis=1), 0)
/home/m3rlin45/.cache/pypoetry/virtualenvs/motorsports-data-notebook-qUYIIvW5-py3.12/lib/python3.12/site-packages/libxrk/gps.py:348: RuntimeWarning: invalid value encountered in multiply
  dist = np.sum(np.square(O + t.reshape((len(t), 1)) * D), axis=1)


In [5]:
def get_best_lap(laps_df):
    """
    Find the best (fastest) lap by duration.
    
    Parameters
    ----------
    laps_df : pandas.DataFrame
        Laps table with 'start_time', 'end_time' columns.
        
    Returns
    -------
    pandas.Series
        The row corresponding to the best lap.
    """
    if not {'start_time','end_time'}.issubset(laps_df.columns):
        raise ValueError("Expected start_time and end_time columns in laps table")
    
    laps_with_duration = laps_df.copy()
    laps_with_duration['lap_duration_ms'] = laps_with_duration['end_time'] - laps_with_duration['start_time']
    best_idx = laps_with_duration['lap_duration_ms'].idxmin()
    return laps_with_duration.loc[best_idx]


# Start line helper
def compute_start_line(lat, lon, ahead_points=100, scale=0.02):
    """
    Compute endpoints for a perpendicular start/finish line at the beginning of the track.
    
    Parameters
    ----------
    lat : pandas.Series
        Latitude values along the track.
    lon : pandas.Series
        Longitude values along the track.
    ahead_points : int, default=100
        Number of points ahead to use for computing the heading direction.
    scale : float, default=0.02
        Line length as a fraction of track size (lon/lat range).
        
    Returns
    -------
    tuple of tuples
        ((lat_a, lon_a), (lat_b, lon_b)) - endpoints of the perpendicular line.
    """
    import math
    lat1 = lat.iloc[0]; lon1 = lon.iloc[0]
    idx2 = min(ahead_points, len(lat)-1)
    lat2 = lat.iloc[idx2]; lon2 = lon.iloc[idx2]
    heading_vec = (lon2 - lon1, lat2 - lat1)
    perp_vec = (-heading_vec[1], heading_vec[0])
    norm = math.hypot(perp_vec[0], perp_vec[1]) or 1.0
    lon_range = lon.max() - lon.min()
    lat_range = lat.max() - lat.min()
    half_len_deg = scale * max(lon_range, lat_range)
    dx = perp_vec[0] / norm * half_len_deg
    dy = perp_vec[1] / norm * half_len_deg
    return (lat1 - dy, lon1 - dx), (lat1 + dy, lon1 + dx)


def plot_lap_gps(lat, lon, color_values, color_label=None,
                 width=800, height=800, colorscale='Viridis', title=None):
    """
    Plot interactive GPS track with color-coded data and perpendicular start line using Plotly.
    
    Parameters
    ----------
    lat : pandas.Series or array-like
        Latitude values along the track.
    lon : pandas.Series or array-like
        Longitude values along the track.
    color_values : pandas.Series or array-like
        Values to use for color mapping.
    color_label : str, optional
        Label for the colorbar. If None, no label is shown.
    width : int, default=800
        Figure width in pixels.
    height : int, default=800
        Figure height in pixels.
    colorscale : str, default='Viridis'
        Plotly colorscale name (e.g., 'Viridis', 'Plasma', 'Jet', 'Rainbow').
    title : str, optional
        Plot title. If None, uses color_label in title if available.
        
    Returns
    -------
    plotly.graph_objs.Figure
        Interactive Plotly figure object.
    """
    # Compute start line
    (lat_a, lon_a), (lat_b, lon_b) = compute_start_line(lat, lon)
    
    # Create interactive Plotly figure
    fig = go.Figure()
    
    # Add GPS track colored by specified values
    fig.add_trace(go.Scattergl(
        x=lon,
        y=lat,
        mode='markers',
        marker=dict(
            size=4,
            color=color_values,
            colorscale=colorscale,
            showscale=True,
            colorbar=dict(title=color_label) if color_label else dict()
        ),
        name='GPS Track',
        showlegend=False,
        hovertemplate=f'{color_label or "Value"}: %{{marker.color:.2f}}<br>Lon: %{{x:.6f}}<br>Lat: %{{y:.6f}}<extra></extra>'
    ))
    
    # Add track outline
    fig.add_trace(go.Scattergl(
        x=lon,
        y=lat,
        mode='lines',
        line=dict(color='black', width=1),
        opacity=0.3,
        name='Track Outline',
        showlegend=False,
        hoverinfo='skip'
    ))
    
    # Add start/finish line
    fig.add_trace(go.Scattergl(
        x=[lon_a, lon_b],
        y=[lat_a, lat_b],
        mode='lines',
        line=dict(color='black', width=3),
        name='Start/Finish',
        showlegend=False,
        hoverinfo='skip'
    ))
    
    # Update layout
    fig.update_layout(
        title=title or (f'GPS Path ({color_label})' if color_label else 'GPS Path'),
        width=width,
        height=height,
        xaxis=dict(
            scaleanchor='y',
            scaleratio=1,
            showticklabels=False,
            showgrid=False,
            zeroline=False
        ),
        yaxis=dict(
            showticklabels=False,
            showgrid=False,
            zeroline=False
        ),
        plot_bgcolor='white',
        hovermode='closest'
    )
    
    return fig


In [7]:
# Best lap extraction and interactive Plotly GPS speed plot (km/h)
# Get best lap
best_lap = get_best_lap(laps)
start_ts = best_lap['start_time']
end_ts = best_lap['end_time']
channels['speed_kmh'] = channels['GPS Speed'] * 3.6
lap_channels = channels.query(f'timecodes >= @start_ts and timecodes <= @end_ts').copy()

# Plot
fig = plot_lap_gps(
    lat=lap_channels['GPS Latitude'],
    lon=lap_channels['GPS Longitude'],
    color_values=lap_channels['speed_kmh'],
    color_label='Speed (km/h)',
    title='Best Lap GPS Path'
)
fig.show()
